<a href="https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/Subhash-2910/flyrank-ML-T1.git"
REPO_DIR = "flyrank-ML-T1"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working directory:", os.getcwd())

assert os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
), "Starter CSV not found — are you at the repository root?"

print("Starter data has been found. You're ready.")

Working directory: /content/flyrank-ML-T1
Starter data has been found. You're ready.


## 1. My rule and its reason codes




## Rule

I will prioritize pages for human refresh review when they are both **stale** and **highly visible in organic search**.

This is a decision-support rule, not a guarantee that refreshing a page will improve traffic or rankings. A page may be old and visible because it is evergreen, seasonal, intentionally unchanged, or already accurate.

## Signals checked first

1. **Staleness/content age** — a signal linked to FlyRank refresh logic.
2. **Current 90-day impressions** — a measure of current organic-search visibility and the audience potentially affected by a review.

## Rule output

- **Score:** 60% staleness plus 40% current visibility
- **Reason code:** `STALE_VISIBLE_PAGE`
- **Action label:** `REVIEW_FOR_REFRESH`

## Signal verdicts

- **Staleness verdict: MIXED.** Staleness alone does not prove a page needs work, so I will combine it with current search visibility.
- **Visibility verdict: CONFIRMED.** Higher current impressions indicate more observed search exposure at stake, so visibility is useful for prioritizing human review. It does not guarantee that a refresh will help.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

id_col = "content_id"
impressions_col = "impressions_90d"
trend_col = "trend_pct"

# Finds the age/freshness column.
age_candidates = [
    "days_since_update",
    "days_since_last_update",
    "content_age_days",
    "age_days",
    "freshness_days",
    "days_since_refresh",
]

age_col = next((col for col in age_candidates if col in df.columns), None)

if age_col is None:
    raise ValueError(
        "Age/freshness column not found. Open docs/data-dictionary.md, "
        "find the correct age/freshness column, and add it to age_candidates."
    )

if trend_col not in df.columns:
    raise ValueError(
        "trend_pct not found. Check docs/data-dictionary.md and replace "
        "trend_col with the correct historical trend column."
    )

print(f"\nUsing age/freshness column: {age_col}")
print(f"Using trend column: {trend_col}")

# Signal check 1: staleness, including n.
staleness_check = (
    df.assign(staleness_bucket=pd.qcut(df[age_col], q=4, duplicates="drop"))
    .groupby("staleness_bucket", observed=True)
    .agg(
        n=(id_col, "size"),
        median_impressions_90d=(impressions_col, "median"),
        mean_historical_trend_pct=(trend_col, "mean"),
    )
    .round(2)
)

# Signal check 2: visibility, including n.
visibility_check = (
    df.assign(visibility_bucket=pd.qcut(df[impressions_col], q=4, duplicates="drop"))
    .groupby("visibility_bucket", observed=True)
    .agg(
        n=(id_col, "size"),
        median_age_days=(age_col, "median"),
        mean_historical_trend_pct=(trend_col, "mean"),
    )
    .round(2)
)

print("Signal check 1: Staleness/content age")
display(staleness_check)

print("Signal check 2: Current 90-day impressions")
display(visibility_check)

Dataset shape: (30000, 44)

Columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Using age/freshness column: days_since_last_update
Using trend column: trend_pct
Signal check 1: Staleness/content age


,n,median_impressions_90d,mean_historical_trend_pct
staleness_bucket,,,
"(0.999, 20.0]",15866,363.0,-0.90
"(20.0, 104.0]",13816,1262.0,-8.86
"(104.0, 373.0]",318,30.0,11.21


Signal check 2: Current 90-day impressions


,n,median_age_days,mean_historical_trend_pct
visibility_bucket,,,
"(0.999, 81.0]",7503,20.0,-6.89
"(81.0, 731.0]",7499,22.0,10.51
"(731.0, 3615.25]",7498,22.0,-9.32
"(3615.25, 517715.0]",7500,25.0,-14.02


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*





The score uses only historical observable inputs available at the decision moment:

- **60% staleness/content age**
- **40% current 90-day impressions**

A page enters the refresh-review queue only when it is in the top 25% for both signals.

Every queued row has one reason code, `STALE_VISIBLE_PAGE`, and one action label, `REVIEW_FOR_REFRESH`.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Thresholds are calculated from this dataset.
stale_cutoff = df[age_col].quantile(0.75)
visible_cutoff = df[impressions_col].quantile(0.75)

# Keep only score inputs and the pseudonymized content ID.
df_scored = df[[id_col, age_col, impressions_col]].copy()

# Transparent 0–100 score:
# 60 points for relative staleness + 40 points for relative visibility.
df_scored["staleness_component"] = (
    df_scored[age_col].rank(pct=True) * 60
)

df_scored["visibility_component"] = (
    df_scored[impressions_col].rank(pct=True) * 40
)

df_scored["baseline_score"] = (
    df_scored["staleness_component"]
    + df_scored["visibility_component"]
).round(1)

# Queue only pages satisfying both rule conditions.
queue = df_scored.loc[
    (df_scored[age_col] >= stale_cutoff)
    & (df_scored[impressions_col] >= visible_cutoff)
].copy()

# One reason code and one action label.
queue["reason_code"] = "STALE_VISIBLE_PAGE"
queue["action_label"] = "REVIEW_FOR_REFRESH"

# Relative confidence within this baseline; not a probability of success.
queue["confidence_note"] = np.where(
    queue["baseline_score"] >= queue["baseline_score"].quantile(0.75),
    "Higher within this baseline; human review still required.",
    "Moderate within this baseline; human review still required."
)

# Sort highest priority first.
queue = queue.sort_values(
    by=["baseline_score", impressions_col, age_col],
    ascending=False
).reset_index(drop=True)

queue_columns = [
    id_col,
    age_col,
    impressions_col,
    "baseline_score",
    "reason_code",
    "action_label",
    "confidence_note",
]

# Write the complete ranked queue.
os.makedirs("work/outputs", exist_ok=True)

output_path = "work/outputs/baseline_action_score.csv"
queue[queue_columns].to_csv(output_path, index=False)

print(f"Staleness cutoff: {stale_cutoff:.2f}")
print(f"Visibility cutoff: {visible_cutoff:.2f}")
print(f"Queue rows written: {len(queue):,}")
print(f"CSV written to: {output_path}")

display(queue[queue_columns].head(20))

Staleness cutoff: 104.00
Visibility cutoff: 3615.25
Queue rows written: 3,198
CSV written to: work/outputs/baseline_action_score.csv


,content_id,days_since_last_update,impressions_90d,baseline_score,reason_code,action_label,confidence_note
0,content_cf56e2e2e282,194,61678,99.2,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
1,content_7368877ea310,194,59472,99.2,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
2,content_a5dbb404bdc2,106,79035,99.1,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
3,content_47b8b12d581e,106,40305,98.5,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
4,content_1bfaa38ff26c,194,25715,98.0,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
5,content_69fad7e6c50c,106,28000,97.9,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
6,content_482aff19e9cc,106,26287,97.7,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
7,content_6ac3ab740bbf,106,22462,97.4,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
8,content_cb7e312f5d32,151,21272,97.4,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...
9,content_ac1d924c6a70,106,21853,97.3,STALE_VISIBLE_PAGE,REVIEW_FOR_REFRESH,Higher within this baseline; human review stil...


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*





The twenty rows below are the highest-ranked pages under this transparent baseline. Each is recommended for human review, not an automatic rewrite.

A recommendation could be wrong if a page is evergreen, seasonal, intentionally stable, already accurate, constrained by search intent, or unlikely to benefit from a refresh despite being old and visible.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
top20_review = queue.head(20).copy()

top20_review["why_it_is_here"] = top20_review.apply(
    lambda row: (
        f"Stale page ({row[age_col]:.0f} age/freshness days) with "
        f"{row[impressions_col]:.0f} current 90-day impressions; "
        "it meets the stale-and-visible rule."
    ),
    axis=1
)

top20_review["what_would_make_it_wrong"] = (
    "The page may be evergreen, seasonal, intentionally unchanged, "
    "already accurate, or unable to benefit from a refresh."
)

display(
    top20_review[
        [
            id_col,
            "action_label",
            "reason_code",
            "baseline_score",
            "confidence_note",
            "why_it_is_here",
            "what_would_make_it_wrong",
        ]
    ]
)

,content_id,action_label,reason_code,baseline_score,confidence_note,why_it_is_here,what_would_make_it_wrong
0,content_cf56e2e2e282,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,99.2,Higher within this baseline; human review stil...,Stale page (194 age/freshness days) with 61678...,"The page may be evergreen, seasonal, intention..."
1,content_7368877ea310,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,99.2,Higher within this baseline; human review stil...,Stale page (194 age/freshness days) with 59472...,"The page may be evergreen, seasonal, intention..."
2,content_a5dbb404bdc2,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,99.1,Higher within this baseline; human review stil...,Stale page (106 age/freshness days) with 79035...,"The page may be evergreen, seasonal, intention..."
3,content_47b8b12d581e,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,98.5,Higher within this baseline; human review stil...,Stale page (106 age/freshness days) with 40305...,"The page may be evergreen, seasonal, intention..."
4,content_1bfaa38ff26c,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,98.0,Higher within this baseline; human review stil...,Stale page (194 age/freshness days) with 25715...,"The page may be evergreen, seasonal, intention..."
5,content_69fad7e6c50c,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,97.9,Higher within this baseline; human review stil...,Stale page (106 age/freshness days) with 28000...,"The page may be evergreen, seasonal, intention..."
6,content_482aff19e9cc,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,97.7,Higher within this baseline; human review stil...,Stale page (106 age/freshness days) with 26287...,"The page may be evergreen, seasonal, intention..."
7,content_6ac3ab740bbf,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,97.4,Higher within this baseline; human review stil...,Stale page (106 age/freshness days) with 22462...,"The page may be evergreen, seasonal, intention..."
8,content_cb7e312f5d32,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,97.4,Higher within this baseline; human review stil...,Stale page (151 age/freshness days) with 21272...,"The page may be evergreen, seasonal, intention..."
9,content_ac1d924c6a70,REVIEW_FOR_REFRESH,STALE_VISIBLE_PAGE,97.3,Higher within this baseline; human review stil...,Stale page (106 age/freshness days) with 21853...,"The page may be evergreen, seasonal, intention..."


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*




Weak picks meet only one part of the rule. They are not promoted to immediate refresh review because the baseline requires both staleness and current visibility.

Leakage check: I used only content age/freshness and current 90-day impressions. I did not use product flags, composite product scores, IDs as features, target-derived fields, or future-window metrics.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks meet only one of the two required conditions.
weak_picks = df_scored.loc[
    (
        ((df_scored[age_col] >= stale_cutoff)
         & (df_scored[impressions_col] < visible_cutoff))
        |
        ((df_scored[age_col] < stale_cutoff)
         & (df_scored[impressions_col] >= visible_cutoff))
    )
].copy()

weak_picks["recommended_action"] = "MONITOR"

weak_picks["why_weak"] = np.where(
    (weak_picks[age_col] >= stale_cutoff)
    & (weak_picks[impressions_col] < visible_cutoff),
    "Stale, but current visibility is below the queue threshold.",
    "Visible, but not stale enough for this refresh-focused rule."
)

display(
    weak_picks[
        [
            id_col,
            age_col,
            impressions_col,
            "baseline_score",
            "recommended_action",
            "why_weak",
        ]
    ]
    .sort_values("baseline_score", ascending=False)
    .head(10)
)

print("Leakage check")
for forbidden in [
    "health_score",
    "needs_ctr_fix",
    "is_quick_win",
    "client_id",
]:
    print(f"{forbidden}: NOT USED AS A FEATURE")

,content_id,days_since_last_update,impressions_90d,baseline_score,recommended_action,why_weak
22004,content_5aa14ed789b2,106,3563,89.4,MONITOR,"Stale, but current visibility is below the que..."
9761,content_47116dd5a563,105,3526,89.3,MONITOR,"Stale, but current visibility is below the que..."
13121,content_cbffee46b629,106,3043,88.5,MONITOR,"Stale, but current visibility is below the que..."
21074,content_701f7861d94a,106,2971,88.4,MONITOR,"Stale, but current visibility is below the que..."
18063,content_b08562686d22,106,2846,88.1,MONITOR,"Stale, but current visibility is below the que..."
2104,content_c3970ef960d5,106,2801,88.0,MONITOR,"Stale, but current visibility is below the que..."
12471,content_7bba5d83959a,106,2724,87.8,MONITOR,"Stale, but current visibility is below the que..."
21945,content_434d25702cdb,106,2410,87.0,MONITOR,"Stale, but current visibility is below the que..."
29866,content_452a4e18212c,106,2243,86.6,MONITOR,"Stale, but current visibility is below the que..."
1783,content_323d0bdf7077,106,2018,85.9,MONITOR,"Stale, but current visibility is below the que..."


Leakage check
health_score: NOT USED AS A FEATURE
needs_ctr_fix: NOT USED AS A FEATURE
is_quick_win: NOT USED AS A FEATURE
client_id: NOT USED AS A FEATURE


In [6]:
print("Self-check complete.")
print(f"CSV exists: {os.path.exists('work/outputs/baseline_action_score.csv')}")
print(f"Top-20 review rows: {len(top20_review)}")

Self-check complete.
CSV exists: True
Top-20 review rows: 20


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.